# Stage 1 teacher/causal anchors through one frozen Step 2

This notebook evaluates any available Stage 1 supplied-gap checkpoint. It is safe to run with only the full-audio teacher or only the causal student on one server. Each run creates a portable bundle; copy the teacher and causal bundles onto either server and list them in `COPIED_BUNDLES` to compare metrics and render both models together.

The matched conditions are: codec reconstruction, GT anchors through Step 2, generated anchors through Step 2, and anchor substitution. Every bundle records and verifies the selected clips, supplied schedule, frozen Step 2 fingerprint, codecs, and decoding protocol.

In [ ]:
from pathlib import Path, PurePosixPath
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Video, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.parent != PROJECT_ROOT and not (PROJECT_ROOT / 'motion_generation').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'motion_generation').is_dir(), PROJECT_ROOT
MOTION_GENERATION_DIR = PROJECT_ROOT / 'motion_generation'
if str(MOTION_GENERATION_DIR) not in sys.path:
    sys.path.insert(0, str(MOTION_GENERATION_DIR))

# Use the active notebook kernel by default. Override if Jupyter uses the wrong environment.
PYTHON_EXECUTABLE = Path(sys.executable)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# Any label and checkpoint path are accepted. Existing defaults are discovered automatically.
DEFAULT_STEP1_CANDIDATES = {
    'teacher': PROJECT_ROOT / 'checkpoints/step1_stage1_anchor_ce_uniform_gap100_nano_q0q3_vocab/best',
    'causal': PROJECT_ROOT / 'checkpoints/step1_stage1_anchor_ce_uniform_gap100_nano_q0q3_vocab_causal/best',
}
STEP1_CHECKPOINT_OVERRIDES = {
    # Example: 'teacher_epoch100': Path('/absolute/path/to/checkpoint'),
}
LOCAL_STEP1_CHECKPOINTS = {
    label: path for label, path in DEFAULT_STEP1_CANDIDATES.items() if path.is_dir()
}
LOCAL_STEP1_CHECKPOINTS.update({
    label: Path(path).expanduser().resolve()
    for label, path in STEP1_CHECKPOINT_OVERRIDES.items()
})

STEP2_CHECKPOINT = PROJECT_ROOT / 'checkpoints/mask_multipart_body_causal_moss_nano_all16_variable_c2f_soft_recovery_sf05_stage2_gap1_15'
STEP2_CONFIG = PROJECT_ROOT / 'motion_generation/configs/audio_c2f_body_causal_moss_nano_all16_soft_recovery_sf05_stage2.yaml'
BUNDLE_ROOT = PROJECT_ROOT / 'motion_generation/outputs/step1_stage1_teacher_causal_step2'

# Copy a complete <label>/ bundle from the other server, then register it here.
# Each bundle must contain stage1/ and motion/ subdirectories.
COPIED_BUNDLES = {
    # 'Teacher (copied)': Path('/path/to/copied/teacher'),
    # 'Causal (copied)': Path('/path/to/copied/causal'),
}

MAX_CLIPS = 128       # 0 = all 635 validation clips
SUBSET_SEED = 42
TEACHER_BATCH_SIZE = 32
ROLLOUT_BATCH_SIZE = 8
STEP2_BATCH_SIZE = 256
NUM_WORKERS = 0

RUN_STAGE1_EXPORT = True
RUN_MOTION_EVALUATION = True
RUN_FID = False       # First validate 128 clips; then set MAX_CLIPS=0 and RUN_FID=True.
RUN_VISUALIZATION = True
VISUAL_MAX_CLIPS = 4
VISUAL_CLIP_NAMES = []  # Optional exact names; empty selects representative clips.
VISUAL_FRAME_STEP = 2

print('project:', PROJECT_ROOT)
print('python:', PYTHON_EXECUTABLE)
print('device:', DEVICE)
print('local Step 1 checkpoints:')
for label, path in LOCAL_STEP1_CHECKPOINTS.items():
    print(f'  {label}: {path}')
print('copied bundles:')
for label, path in COPIED_BUNDLES.items():
    print(f'  {label}: {path}')
if RUN_STAGE1_EXPORT:
    assert LOCAL_STEP1_CHECKPOINTS, 'No local checkpoint was discovered; set STEP1_CHECKPOINT_OVERRIDES.'
assert STEP2_CHECKPOINT.is_dir(), STEP2_CHECKPOINT
assert STEP2_CONFIG.is_file(), STEP2_CONFIG

## 1. Export Stage 1 metrics and matched anchor caches

Each checkpoint is evaluated independently. This keeps the exact same workflow when teacher and causal models are on different servers. The evaluation schedule is deterministic at epoch 0; the exporter writes a SHA-256 hash so copied bundles can be checked before comparison.

In [ ]:
local_bundles = {}
for label, checkpoint in LOCAL_STEP1_CHECKPOINTS.items():
    bundle_dir = BUNDLE_ROOT / label
    stage1_dir = bundle_dir / 'stage1'
    local_bundles[label] = bundle_dir
    command = [
        str(PYTHON_EXECUTABLE),
        str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_stage1_supplied_gap.py'),
        '--checkpoint', f'{label}={checkpoint}',
        '--output_dir', str(stage1_dir),
        '--step2_checkpoint', str(STEP2_CHECKPOINT),
        '--device', DEVICE,
        '--max_clips', str(MAX_CLIPS),
        '--teacher_batch_size', str(TEACHER_BATCH_SIZE),
        '--rollout_batch_size', str(ROLLOUT_BATCH_SIZE),
        '--num_workers', str(NUM_WORKERS),
        '--subset_seed', str(SUBSET_SEED),
    ]
    print('\n' + ' '.join(command))
    if RUN_STAGE1_EXPORT:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    else:
        print('Skipped Stage 1 export; existing files will be used.')

## 2. Run the same frozen Step 2 and decode motion

This evaluates both matched GT anchors and the selected model's generated-history anchors. `RUN_FID=False` still exports Step 2 likelihood, decoded errors, complete body/hand motions, and visualization inputs. Enable FID only after the pilot protocol passes.

In [ ]:
for label, bundle_dir in local_bundles.items():
    stage1_dir = bundle_dir / 'stage1'
    motion_dir = bundle_dir / 'motion'
    report_path = stage1_dir / 'stage1_evaluation_report.json'
    assert report_path.is_file(), report_path
    report = json.loads(report_path.read_text(encoding='utf-8'))
    conditions = list(report['rollout_caches'])
    gt_conditions = [value for value in conditions if value.endswith('_gt_anchors')]
    generated_conditions = [value for value in conditions if value.endswith('_generated_history')]
    assert len(gt_conditions) == 1, gt_conditions
    assert len(generated_conditions) == 1, generated_conditions
    command = [
        str(PYTHON_EXECUTABLE),
        str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_adaptive_motion.py'),
        '--adaptive_output_dir', str(stage1_dir),
        '--output_dir', str(motion_dir),
        '--step2_config', str(STEP2_CONFIG),
        '--step2_checkpoint', str(STEP2_CHECKPOINT),
        '--device', DEVICE,
        '--step2_batch_size', str(STEP2_BATCH_SIZE),
        '--metric_seed', str(SUBSET_SEED),
        '--condition', gt_conditions[0],
        '--condition', generated_conditions[0],
    ]
    if not RUN_FID:
        command.append('--export_only')
    print('\n' + ' '.join(command))
    if RUN_MOTION_EVALUATION:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    else:
        print('Skipped motion evaluation; existing files will be used.')

## 3. Register local and copied bundles, then verify fairness

The comparison stops if clip selection, supplied schedules, or frozen Step 2 weights differ. Paths may differ across servers; fingerprints must not.

In [ ]:
ACTIVE_BUNDLES = {label: path for label, path in local_bundles.items()}
ACTIVE_BUNDLES.update({
    label: Path(path).expanduser().resolve()
    for label, path in COPIED_BUNDLES.items()
})
assert ACTIVE_BUNDLES, 'No local or copied result bundle is configured.'

bundle_info = {}
for display_label, bundle_dir in ACTIVE_BUNDLES.items():
    stage1_dir = bundle_dir / 'stage1'
    motion_dir = bundle_dir / 'motion'
    stage1_report_path = stage1_dir / 'stage1_evaluation_report.json'
    motion_contract_path = motion_dir / 'adaptive_motion_contract.json'
    assert stage1_report_path.is_file(), stage1_report_path
    assert motion_contract_path.is_file(), motion_contract_path
    stage1_report = json.loads(stage1_report_path.read_text(encoding='utf-8'))
    motion_contract = json.loads(motion_contract_path.read_text(encoding='utf-8'))
    generated = [
        value for value in stage1_report['rollout_caches']
        if value.endswith('_generated_history')
    ]
    ground_truth = [
        value for value in stage1_report['rollout_caches']
        if value.endswith('_gt_anchors')
    ]
    assert len(generated) == 1 and len(ground_truth) == 1
    bundle_info[display_label] = {
        'root': bundle_dir,
        'stage1_dir': stage1_dir,
        'motion_dir': motion_dir,
        'stage1_report': stage1_report,
        'motion_contract': motion_contract,
        'generated_condition': generated[0],
        'gt_condition': ground_truth[0],
    }

protocol_keys = (
    'selected_names_sha256', 'schedule_sha256', 'schedule_seed',
    'supplied_gap_distribution', 'supplied_gap_range',
)
reference_label = next(iter(bundle_info))
reference = bundle_info[reference_label]
reference_protocol = reference['stage1_report']['protocol']
reference_step2 = reference['motion_contract']['step2_checkpoint_fingerprint']
for label, info in bundle_info.items():
    protocol = info['stage1_report']['protocol']
    for key in protocol_keys:
        assert protocol[key] == reference_protocol[key], (
            f'{label}: protocol mismatch for {key}: '
            f"{protocol[key]} != {reference_protocol[key]}"
        )
    assert info['motion_contract']['step2_checkpoint_fingerprint'] == reference_step2, (
        f'{label}: frozen Step 2 fingerprint differs'
    )
    assert info['motion_contract']['causal_codec_fingerprints'] == reference['motion_contract']['causal_codec_fingerprints'], (
        f'{label}: causal codec contract differs'
    )

print('PASS: bundles use identical clips, schedules, Step 2 weights, and codecs')
print('schedule:', reference_protocol['schedule_sha256'])
print('Step 2:', reference_step2)
print('bundles:', list(bundle_info))

In [ ]:
def load_optional_csv(path):
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()

table_specs = {
    'Teacher-forced Step 1': ('stage1', 'stage1_teacher_forced.csv'),
    'Generated-history Step 1': ('stage1', 'stage1_generated_rollout.csv'),
    'Step 1 by supplied gap': ('stage1', 'stage1_generated_rollout_by_gap.csv'),
    'Frozen Step 2 likelihood': ('motion', 'step2_c2f_summary.csv'),
    'Decoded motion summary': ('motion', 'decoded_metrics_summary.csv'),
    'Motion FID': ('motion', 'adaptive_motion_fid.csv'),
}
combined_tables = {}
for title, (subdir, filename) in table_specs.items():
    values = []
    for display_label, info in bundle_info.items():
        frame = load_optional_csv(info[f'{subdir}_dir'] / filename)
        if not frame.empty:
            frame.insert(0, 'bundle', display_label)
            values.append(frame)
    combined = pd.concat(values, ignore_index=True) if values else pd.DataFrame()
    combined_tables[title] = combined
    print('\n' + title)
    display(combined)


In [ ]:
# Compact plots for the two decisive questions: anchor prediction and final motion.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
teacher_frame = combined_tables['Teacher-forced Step 1']
rollout_frame = combined_tables['Generated-history Step 1']
if not teacher_frame.empty and not rollout_frame.empty:
    anchor_plot = teacher_frame[['bundle', 'accuracy']].rename(columns={'accuracy': 'teacher_forced'})
    anchor_plot = anchor_plot.merge(
        rollout_frame[['bundle', 'accuracy']].rename(columns={'accuracy': 'generated_history'}),
        on='bundle', how='inner',
    ).set_index('bundle')
    anchor_plot.plot.bar(ax=axes[0], color=['#4c78a8', '#e45756'])
    axes[0].set_title('Step 1 anchor token accuracy')
    axes[0].set_ylabel('accuracy')
    axes[0].tick_params(axis='x', rotation=20)
else:
    axes[0].text(0.5, 0.5, 'Step 1 tables unavailable', ha='center')

decoded = combined_tables['Decoded motion summary']
if not decoded.empty:
    generated_rows = []
    for label, info in bundle_info.items():
        selected = decoded[
            decoded['bundle'].eq(label)
            & decoded['protocol'].eq('step2_infilled')
            & decoded['condition'].eq(info['generated_condition'])
        ]
        generated_rows.append(selected)
    final_motion = pd.concat(generated_rows, ignore_index=True)
    final_motion.set_index('bundle')['raw_gt_rmse'].plot.bar(ax=axes[1], color='#72b7b2')
    axes[1].set_title('Generated anchors + frozen Step 2')
    axes[1].set_ylabel('decoded RMSE vs raw GT (lower is better)')
    axes[1].tick_params(axis='x', rotation=20)
else:
    axes[1].text(0.5, 0.5, 'Decoded motion table unavailable', ha='center')
fig.tight_layout()
display(fig)
plt.close(fig)

## 4. Select representative clips and render synchronized motion

Automatic selection includes a median-error clip, a difficult clip, and—when two models are present—the clip with the largest teacher/causal RMSE difference. Set `VISUAL_CLIP_NAMES` to override this. The video panels are raw GT, codec reconstruction, GT anchors through Step 2, then every registered model through Step 2.

In [ ]:
from models.step1_mimi_planner import canonical_data_path
from utils.face_infill_visualization import (
    plot_tiled_full_clip_summary,
    prepare_exported_full_clip_comparison,
    save_tiled_full_clip_video,
)

reference_manifest = pd.read_csv(reference['motion_dir'] / 'motion_manifest.csv')
schedule_rows = json.loads((reference['stage1_dir'] / 'schedule_manifest.json').read_text(encoding='utf-8'))
schedule_by_name = {row['name']: row for row in schedule_rows}
stem_by_name = {
    row['name']: f"{int(row['clip_index']):06d}"
    for _, row in reference_manifest.iterrows()
}

per_model_errors = []
for label, info in bundle_info.items():
    frame = pd.read_csv(info['motion_dir'] / 'decoded_metrics_per_clip.csv')
    frame = frame[
        frame['protocol'].eq('step2_infilled')
        & frame['condition'].eq(info['generated_condition'])
    ][['name', 'raw_gt_rmse']].rename(columns={'raw_gt_rmse': label})
    per_model_errors.append(frame)
error_table = per_model_errors[0]
for frame in per_model_errors[1:]:
    error_table = error_table.merge(frame, on='name', validate='one_to_one')
model_columns = list(bundle_info)
error_table['mean_model_rmse'] = error_table[model_columns].mean(axis=1)

if VISUAL_CLIP_NAMES:
    selected_visual_names = list(VISUAL_CLIP_NAMES)
else:
    candidates = []
    median = float(error_table['mean_model_rmse'].median())
    candidates.append(error_table.loc[(error_table['mean_model_rmse'] - median).abs().idxmin(), 'name'])
    candidates.append(error_table.loc[error_table['mean_model_rmse'].idxmax(), 'name'])
    candidates.append(error_table.loc[error_table['mean_model_rmse'].idxmin(), 'name'])
    if len(model_columns) >= 2:
        difference = (error_table[model_columns[0]] - error_table[model_columns[1]]).abs()
        candidates.append(error_table.loc[difference.idxmax(), 'name'])
    selected_visual_names = list(dict.fromkeys(candidates))[:VISUAL_MAX_CLIPS]

assert all(name in stem_by_name for name in selected_visual_names), selected_visual_names
display(error_table[error_table['name'].isin(selected_visual_names)].sort_values('mean_model_rmse'))
print('selected:', selected_visual_names)

In [ ]:
TEMPLATE_BVH = PROJECT_ROOT / 'motion_generation/meta/template_susu_retarget_63nodes.bvh'
WAV_DIR = PROJECT_ROOT / 'SuSuInterActs/SuSuInterActs/wav_data'
VISUAL_OUTPUT_DIR = BUNDLE_ROOT / 'combined_visualizations'
VISUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert TEMPLATE_BVH.is_file(), TEMPLATE_BVH
visual_device = torch.device(DEVICE)

rendered = []
for name in selected_visual_names:
    stem = stem_by_name[name]
    variants = {
        'Raw GT': reference['motion_dir'] / 'visual_motion/raw_gt' / f'{stem}.npy',
        'Codec GT': reference['motion_dir'] / 'visual_motion/causal_codec_reconstruction' / f'{stem}.npy',
        'Step2 + GT anchors': (
            reference['motion_dir'] / 'visual_motion/step2_infilled'
            / reference['gt_condition'] / f'{stem}.npy'
        ),
    }
    for label, info in bundle_info.items():
        variants[f'{label} + Step2'] = (
            info['motion_dir'] / 'visual_motion/step2_infilled'
            / info['generated_condition'] / f'{stem}.npy'
        )
    for label, path in variants.items():
        assert Path(path).is_file(), f'{label}: {path}'
    comparison = prepare_exported_full_clip_comparison(
        name=name,
        motion_variants=variants,
        anchor_times=schedule_by_name[name]['anchor_times'],
        template_bvh=TEMPLATE_BVH,
        device=visual_device,
        fps=20,
    )
    safe_name = '__'.join(PurePosixPath(name).parts[-2:]).replace(' ', '_')
    png_path = VISUAL_OUTPUT_DIR / f'{safe_name}.png'
    mp4_path = VISUAL_OUTPUT_DIR / f'{safe_name}.mp4'
    plot_tiled_full_clip_summary(comparison, output_path=png_path)
    audio_path = canonical_data_path(WAV_DIR, name, '.wav')
    if RUN_VISUALIZATION:
        save_tiled_full_clip_video(
            comparison,
            mp4_path,
            audio_path=audio_path if audio_path.is_file() else None,
            frame_step=VISUAL_FRAME_STEP,
        )
    rendered.append({'name': name, 'summary': png_path, 'video': mp4_path})
    display(Image(filename=str(png_path)))
    if mp4_path.is_file():
        display(Video(str(mp4_path), embed=False))

pd.DataFrame(rendered)

## Interpretation order

1. Compare teacher-forced Step 1 CE/accuracy to test whether full future audio improves local next-anchor prediction.
2. Compare generated-history accuracy to measure autoregressive error accumulation.
3. Compare frozen Step 2 CE under GT versus generated anchors; this directly measures endpoint usefulness.
4. Use decoded RMSE/velocity/acceleration/jerk and Step 2-infilling FID for final motion quality.
5. Treat anchor-substitution FID only as an anchor diagnostic; it is not the deployed pipeline.

The full-audio teacher and causal student also differ in attention layout, so a teacher advantage is a practical teacher-versus-online result—not a perfectly isolated future-audio ablation.